# AI & Automation Hackathon — Company Research Pipeline

This notebook is self-contained. Run every cell top to bottom.
When prompted, paste a JSON array of company URLs, e.g.:

```
["https://www.example-tech-solutions.com", "https://www.sample-logistics-corp.com"]
```

The notebook scrapes each site (multi-approach: sitemap discovery, homepage link crawl with fuzzy
matching for About/Contact/Services pages, homepage-only fallback), strips boilerplate to cut token
usage, calls Claude to extract a structured company profile, cross-checks emails/phone numbers against
the scraped text to prevent hallucination, and prints a JSON array matching the required schema.

In [ ]:
!pip install -q requests beautifulsoup4 lxml anthropic

In [ ]:
import json
import re
import time
from datetime import datetime, timezone
from difflib import SequenceMatcher
from getpass import getpass
from urllib.parse import urljoin, urlparse
from xml.etree import ElementTree

import requests
from bs4 import BeautifulSoup
import anthropic

ANTHROPIC_API_KEY = getpass('Anthropic API key: ')
MODEL_NAME = 'claude-sonnet-5'

REQUEST_TIMEOUT = 10
REQUEST_RETRIES = 2
REQUEST_DELAY = 0.6
MAX_PAGES_PER_SITE = 5
MAX_CHARS_PER_PAGE = 6000
MAX_TOTAL_CHARS = 18000
MAX_OUTPUT_TOKENS = 1200
USER_AGENT = 'Mozilla/5.0 (compatible; CompanyInsightBot/1.0; +https://example.com/bot)'

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
session = requests.Session()
session.headers.update({'User-Agent': USER_AGENT})

## Smart link discovery (sitemap + fuzzy matching, no blind crawling)

In [ ]:
KEYWORDS = {
    'about': ['about', 'about-us', 'aboutus', 'company', 'who-we-are', 'our-story', 'overview'],
    'contact': ['contact', 'contact-us', 'contactus', 'get-in-touch', 'reach-us', 'support'],
    'services': ['service', 'services', 'solutions', 'products', 'what-we-do', 'offerings', 'platform'],
}
ALL_KEYWORDS = [kw for group in KEYWORDS.values() for kw in group]
SKIP_EXTENSIONS = ('.pdf', '.jpg', '.jpeg', '.png', '.gif', '.svg', '.zip', '.css', '.js',
                    '.xml', '.mp4', '.mp3', '.ico', '.webp', '.woff', '.woff2')


def same_domain(url, base_netloc):
    return urlparse(url).netloc.lower().lstrip('www.') == base_netloc.lower().lstrip('www.')


def fetch_sitemap_urls(base_url):
    base_netloc = urlparse(base_url).netloc
    for sitemap_url in (urljoin(base_url, '/sitemap.xml'), urljoin(base_url, '/sitemap_index.xml')):
        xml_text = fetch_url(sitemap_url)
        if not xml_text:
            continue
        try:
            root = ElementTree.fromstring(xml_text)
        except ElementTree.ParseError:
            continue
        locs = [el.text.strip() for el in root.iter() if el.tag.endswith('loc') and el.text]
        urls = [loc for loc in locs if same_domain(loc, base_netloc) and not loc.lower().endswith(SKIP_EXTENSIONS)]
        if urls:
            return urls
    return []


def extract_links(html, base_url):
    soup = BeautifulSoup(html, 'lxml')
    base_netloc = urlparse(base_url).netloc
    seen, links = set(), []
    for anchor in soup.find_all('a', href=True):
        href = anchor['href'].strip()
        if not href or href.startswith(('#', 'mailto:', 'tel:', 'javascript:')):
            continue
        full_url = urljoin(base_url, href).split('#')[0]
        if full_url in seen or full_url.lower().endswith(SKIP_EXTENSIONS):
            continue
        if not same_domain(full_url, base_netloc):
            continue
        seen.add(full_url)
        links.append((anchor.get_text(' ', strip=True), full_url))
    return links


def score_link(text, href):
    haystack = f'{text} {href}'.lower()
    best = 0.0
    for keyword in ALL_KEYWORDS:
        if keyword in haystack:
            best = max(best, 0.85 + 0.15 * (len(keyword) / max(len(haystack), 1)))
            continue
        best = max(best, SequenceMatcher(None, keyword, haystack).ratio())
    return best


def pick_relevant_pages(base_url, homepage_html, limit=MAX_PAGES_PER_SITE):
    ranked, seen = [], {base_url}
    scored_sitemap = sorted(((score_link(u, u), u) for u in fetch_sitemap_urls(base_url)), reverse=True)
    for score, url in scored_sitemap:
        if score >= 0.4 and url not in seen:
            ranked.append(url); seen.add(url)
    scored_links = sorted(((score_link(t, h), h) for t, h in extract_links(homepage_html, base_url)), reverse=True)
    for score, href in scored_links:
        if score >= 0.4 and href not in seen:
            ranked.append(href); seen.add(href)
    return ranked[:limit]

## Multi-approach scraper (sitemap discovery -> link crawl -> homepage-only fallback)

In [ ]:
_last_request_at = {}


def throttle(host):
    last = _last_request_at.get(host, 0.0)
    wait = REQUEST_DELAY - (time.monotonic() - last)
    if wait > 0:
        time.sleep(wait)
    _last_request_at[host] = time.monotonic()


def fetch_url(url, timeout=REQUEST_TIMEOUT, retries=REQUEST_RETRIES):
    host = urlparse(url).netloc
    for attempt in range(retries + 1):
        throttle(host)
        try:
            response = session.get(url, timeout=timeout, allow_redirects=True)
        except requests.RequestException:
            time.sleep(min(2 ** attempt, 4))
            continue
        if response.status_code == 429 or response.status_code >= 500:
            time.sleep(min(2 ** attempt, 4))
            continue
        if response.status_code >= 400:
            return None
        content_type = response.headers.get('Content-Type', '')
        if content_type and 'text' not in content_type and 'xml' not in content_type:
            return None
        return response.text
    return None


def normalize_url(raw_url):
    raw_url = raw_url.strip()
    if not raw_url:
        return raw_url
    if not raw_url.lower().startswith(('http://', 'https://')):
        raw_url = f'https://{raw_url}'
    return raw_url.rstrip('/')


def gather_site_pages(base_url, limit=MAX_PAGES_PER_SITE):
    homepage_html = fetch_url(base_url)
    if not homepage_html:
        return {}
    pages = {base_url: homepage_html}
    for url in pick_relevant_pages(base_url, homepage_html, limit=limit):
        if len(pages) >= limit:
            break
        html = fetch_url(url)
        if html:
            pages[url] = html
    return pages

## Boilerplate stripping + token-budget-aware chunking

In [ ]:
STRIP_TAGS = ['script', 'style', 'noscript', 'nav', 'footer', 'header', 'svg', 'iframe', 'form', 'aside', 'button']
WHITESPACE_RE = re.compile(r'[ \t]+')


def strip_boilerplate(html):
    soup = BeautifulSoup(html, 'lxml')
    for tag_name in STRIP_TAGS:
        for tag in soup.find_all(tag_name):
            tag.decompose()
    return soup.get_text(separator='\n')


def dedupe_lines(text):
    lines = [WHITESPACE_RE.sub(' ', line).strip() for line in text.split('\n')]
    lines = [line for line in lines if line]
    seen, unique_lines = set(), []
    for line in lines:
        key = line.lower()
        if key in seen:
            continue
        seen.add(key)
        unique_lines.append(line)
    return '\n'.join(unique_lines)


def truncate(text, max_chars):
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(' ', 1)[0] + ' …'


def clean_page(html, max_chars):
    return truncate(dedupe_lines(strip_boilerplate(html)), max_chars)


def build_context(pages, max_chars_per_page=MAX_CHARS_PER_PAGE, max_total_chars=MAX_TOTAL_CHARS):
    blocks, budget = [], max_total_chars
    for url, html in pages.items():
        if budget <= 0:
            break
        cleaned = clean_page(html, min(max_chars_per_page, budget))
        if not cleaned:
            continue
        block = f'SOURCE: {url}\n{cleaned}'
        blocks.append(block)
        budget -= len(block)
    return '\n\n---\n\n'.join(blocks)

## Output schema + hallucination-safe AI extraction

In [ ]:
SCHEMA_KEYS = ['website_name', 'company_name', 'address', 'mobile_number', 'mail',
               'core_service', 'target_customer', 'probable_pain_point', 'outreach_opener']


def empty_profile():
    return {k: ([] if k == 'mail' else '') for k in SCHEMA_KEYS}


def coerce_profile(data):
    if not isinstance(data, dict):
        return empty_profile()
    cleaned = {}
    for key in SCHEMA_KEYS:
        value = data.get(key)
        if key == 'mail':
            if isinstance(value, str):
                cleaned[key] = [value] if value.strip() else []
            elif isinstance(value, list):
                cleaned[key] = [str(v).strip() for v in value if str(v).strip()]
            else:
                cleaned[key] = []
        else:
            cleaned[key] = str(value).strip() if value is not None else ''
    return cleaned


SYSTEM_PROMPT = '''You are a precise B2B research analyst that extracts company information strictly from the provided website text.

Rules:
- Use ONLY facts present in the SOURCE text below. Never invent, guess, or infer contact details, names, or services that are not explicitly present.
- If a field cannot be found in the text, return an empty string "" for text fields or an empty array [] for the mail field. Do not use placeholders like "N/A" or "unknown".
- Emails must be copied exactly as they appear in the text.
- Phone numbers must be copied exactly as they appear in the text.
- outreach_opener, probable_pain_point and target_customer may be reasonable business inferences drawn from the described services, but must not fabricate facts (numbers, client names, claims) not implied by the text.
- Respond with ONLY a single valid JSON object matching the exact schema below. No markdown fences, no explanation.

Schema:
{"website_name": string, "company_name": string, "address": string, "mobile_number": string, "mail": string[], "core_service": string, "target_customer": string, "probable_pain_point": string, "outreach_opener": string}'''

EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')


def parse_json_response(raw_text):
    text = raw_text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(json)?', '', text).rsplit('```', 1)[0].strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return {}


def verify_against_source(profile, source_text):
    source_lower = source_text.lower()
    verified_mail = [m for m in profile.get('mail', []) if m.lower() in source_lower]
    profile['mail'] = verified_mail or EMAIL_RE.findall(source_text)[:5]
    phone = profile.get('mobile_number', '')
    if phone:
        digits = re.sub(r'\D', '', phone)
        if digits and digits not in re.sub(r'\D', '', source_text):
            profile['mobile_number'] = ''
    return profile


def call_llm(context_text, source_url):
    user_prompt = f'Target website: {source_url}\n\nSOURCE TEXT:\n{context_text}'
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=MAX_OUTPUT_TOKENS,
        temperature=0,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user_prompt}],
    )
    return ''.join(block.text for block in response.content if block.type == 'text')


def extract_profile(context_text, source_url):
    if not context_text.strip():
        return empty_profile()
    try:
        raw_text = call_llm(context_text, source_url)
    except anthropic.APIError as exc:
        print(f'  AI extraction failed for {source_url}: {exc}')
        return empty_profile()
    profile = coerce_profile(parse_json_response(raw_text))
    return verify_against_source(profile, context_text)

## Required entry point: `enrich_companies(urls) -> list[dict]`

In [ ]:
def enrich_companies(urls):
    results = []
    for raw_url in urls:
        url = normalize_url(raw_url)
        print(f'Processing {url} ...')
        if not url:
            results.append(empty_profile())
            continue
        pages = gather_site_pages(url)
        if not pages:
            print(f'  Could not reach {url}, returning empty schema-stable profile.')
            results.append(empty_profile())
            continue
        context_text = build_context(pages)
        profile = extract_profile(context_text, url)
        results.append(profile)
    return results

## Run it

Paste a JSON array of company URLs when prompted, e.g.
`["https://www.example-tech-solutions.com", "https://www.sample-logistics-corp.com"]`

In [ ]:
raw_input_urls = input('Paste a JSON array of company URLs: ')
urls = json.loads(raw_input_urls)

output = enrich_companies(urls)
print(json.dumps(output, indent=2))

with open('results.json', 'w') as f:
    json.dump(output, f, indent=2)
print('\nSaved to results.json')